# 01 — Auditoría del dataset

**Proyecto:** Spotify Music Intelligence
**Módulo 1:** Ingesta, auditoría y validación
**Objetivo:** verificar de forma reproducible la estructura y calidad de `data/raw/dataset.csv` **sin modificarlo**.

In [ ]:
from pathlib import Path

from spotify_intelligence.data.audit import run_audit

## 1. Carga y auditoría

El script `run_audit` calcula el hash SHA-256, valida columnas y rangos, mide nulos y duplicados, detecta anomalías y comprueba los bloques de género. Guarda el reporte en `reports/data_quality/data_quality_report.json`.

In [ ]:
report = run_audit()
report_path = Path("reports/data_quality/data_quality_report.json")
print(f"Reporte guardado en: {report_path.resolve()}")

In [ ]:
print(f"Dataset hash SHA-256: {report['dataset_hash']}")
print(f"Filas: {report['raw_rows']} | Columnas: {report['raw_columns']}")
print(f"Columnas requeridas OK: {report['required_columns_ok']}")

## 2. Estructura por bloques y prevalencia

El CSV contiene 114 bloques consecutivos de 1.000 filas. Cada bloque corresponde al valor de `track_genre`. Cada género ocupa artificialmente:

```
1.000 / 114.000 = 0,0087719 ≈ 0,877 % del archivo
```

Esto **no** permite estimar la prevalencia real de géneros en Spotify. Solo permite analizar el conjunto balanceado entregado.

In [ ]:
genres = report["genres"]
print(f"Géneros únicos: {genres['total_unique']} (esperado: {genres['expected_unique']})")
print(f"Filas por género iguales en todos los bloques: {genres['genre_counts_all_equal']}")

## 3. Calidad: nulos, duplicados e identidad

- `Unnamed: 0` es un índice artificial y se elimina en datos procesados.
- El dataset original nunca se modifica.

In [ ]:
nulls = report["nulls"]
print(f"Celdas nulas totales: {nulls['total_cells']}")
for col, n in nulls["columns_with_nulls"].items():
    print(f"  {col}: {n}")

In [ ]:
print(f"Track IDs únicos: {report['track_ids_unique']}")
print(f"Filas adicionales respecto a IDs únicos: {report['rows_minus_unique_ids']}")
mg = report["multi_genre"]
print(f"Canciones con más de un género: {mg['tracks_with_multiple_genres']}")
print(f"Máximo de géneros por canción: {mg['max_genres_per_track']}")

## 4. Anomalías

Reglas de interpretación:
- `popularity = 0` se conserva y no significa falta de calidad.
- Duraciones extremas se conservan y se marcan; no son errores automáticos.
- `tempo = 0` forma parte de un patrón de análisis acústico incompleto.

In [ ]:
print(f"Popularidad igual a cero: {report['popularity']['zero_count']}")
print(f"Tracks < 60s: {report['duration']['short_tracks_under_60s']}")
print(f"Tracks > 10min: {report['duration']['long_tracks_over_10min']}")
print(f"Tempo = 0: {report['tempo_zero_count']}")
print(f"Análisis acústico incompleto: {report['incomplete_audio']['count']}")

In [ ]:
violations = report["range_violations"]
if violations:
    print("Violaciones de rango por columna:")
    for col, rows in violations.items():
        print(f"  {col}: {len(rows)} ejemplos")
else:
    print("Sin violaciones de rango.")

## 5. Limitaciones

No puede concluirse:
- Qué proporción real del catálogo de Spotify pertenece a cada género.
- Qué género es el más abundante o escuchado en Spotify.
- Participación de mercado.
- Tendencias globales o temporales.

Sí puede estudiarse: perfiles acústicos, distribuciones, correlaciones, solapamiento entre etiquetas, calidad y separabilidad de etiquetas.